# GPT-SoVITS v2 · 通用 Colab 训练模板

主要输入是一个**已切片的 WAV 语音素材 ZIP**，支持 ZIP 中有子目录。在第一格填写 ZIP 相对于 MyDrive 的路径，例如 `GPT_SoVITS/voice_dataset.zip`，程序自动补全路径。长录播请先切片。所有音频应来自目标说话人；分离和 ASR 完成后仍需抽查音频、核对文字。

默认请求 A100，实际 GPU 由 Colab 分配。先在「运行时 → 更改运行时类型」确认 GPU，再按顺序运行。Drive 在开始时挂载，独立 Miniforge 安装不会主动重启运行时。挂载失败可直接上传 ZIP；此时所有结果为临时文件，请及时下载导出包。

本 Notebook 固定已经核对过的 GPT-SoVITS 提交、v2 底模和 PyTorch CUDA 12.6 组合。新的素材包使用独立实验目录，禁止将不同数据混入旧实验。

### 开头输入项说明

正常使用只需填写素材包，其他输入项可保持默认。

| 变量 | 中文名称 | 用途与默认值 |
| --- | --- | --- |
| `DATASET_ZIP` | 素材 ZIP 的 MyDrive 相对路径 | 必填，默认留空。例如 `GPT_SoVITS/voice_dataset.zip`；MyDrive 根目录的文件只填文件名。不必写 `MyDrive` 或 `/content/drive/MyDrive/`。 |
| `EXPERIMENT_NAME` | 实验名称 | 可留空，由 ZIP 名称自动生成，用于隔离输出目录和命名权重。自定义名称使用英文、数字、下划线或短横线。 |
| `DRIVE_FOLDER` | Drive 备份与缓存目录 | MyDrive 下的相对目录，默认 `GPT_SoVITS`。用于训练结果和缓存，不影响素材 ZIP 的查找路径。 |
| `USE_DRIVE` | 使用 Google Drive | 默认开启，开始时挂载并保存备份；关闭或挂载失败时使用临时存储，需要及时下载 ZIP。 |
| `USE_DRIVE_CACHE` | 复用 Drive 环境与模型缓存 | 默认开启。首次成功后打包保存环境和模型，后续会话先恢复缓存，减少重新安装和下载；缓存目录跨实验共用。 |
| `SEPARATE_BGM` | 分离背景音乐 | 默认开启 BS-Roformer 分离；素材已是干净人声时可以关闭。 |
| `EPOCHS_S2` | SoVITS 训练轮数 | 声学模型训练轮数，默认 15。 |
| `EPOCHS_S1` | GPT 训练轮数 | 韵律模型训练轮数，默认 15。 |
| `BATCH_SIZE` | 每批训练样本数 | 默认 0 表示自动：A100 使用 8，其他 GPU 使用 4。显存不足可改小。 |
| `SAVE_EVERY` | 权重保存间隔 | 默认期望每 5 轮保存；实际间隔取各模型轮数与本值的最大公约数，确保最后一轮保存。 |
| `AUTO_DELETE_RUNTIME` | 完成后自动断开并删除运行时 | 默认关闭。开启后，成功导出时将完整 ZIP 写入 Drive、校验并等待上传完成，再释放运行时；未连接 Drive 或保存失败时保留运行时。 |

In [ ]:
#@title 1. 素材与训练配置，首先连接 Drive
#@markdown ### 素材与实验
#@markdown **素材 ZIP 的 MyDrive 相对路径 · DATASET_ZIP（必填）**：例如 `GPT_SoVITS/voice_dataset.zip`，根目录文件只填文件名。程序自动补全 `/content/drive/MyDrive/`；复制的反斜杠会自动转换。
DATASET_ZIP = "" #@param {type:"string"}
#@markdown **实验名称 · EXPERIMENT_NAME（选填）**：留空会按素材包名自动生成；用于输出目录和模型文件名。自定义时使用英文、数字、下划线或短横线。
EXPERIMENT_NAME = "" #@param {type:"string"}
#@markdown **Drive 备份与缓存目录 · DRIVE_FOLDER**：填写 MyDrive 下的相对目录，例如 `GPT_SoVITS`。用于保存结果和缓存，素材位置由上面的 DATASET_ZIP 单独指定。
DRIVE_FOLDER = "GPT_SoVITS" #@param {type:"string"}
#@markdown **使用 Google Drive · USE_DRIVE**：开启则在最开始挂载 Drive。关闭或挂载失败时使用临时目录，完成后请下载导出 ZIP。
USE_DRIVE = True #@param {type:"boolean"}
#@markdown **复用 Drive 环境与模型缓存 · USE_DRIVE_CACHE**：首次成功后缓存独立 Python 环境、底模、ASR 和分离模型；后续运行从 Drive 恢复到本地。关闭时使用普通安装流程。
USE_DRIVE_CACHE = True #@param {type:"boolean"}
#@markdown **分离背景音乐 · SEPARATE_BGM**：开启 BS-Roformer 人声分离。素材已经清洗为干净人声时可关闭。
SEPARATE_BGM = True #@param {type:"boolean"}
#@markdown ### 训练参数（通常保持默认）
#@markdown **SoVITS 声学模型训练轮数 · EPOCHS_S2**：默认 15 轮。
EPOCHS_S2 = 15 #@param {type:"integer"}
#@markdown **GPT 韵律模型训练轮数 · EPOCHS_S1**：默认 15 轮。
EPOCHS_S1 = 15 #@param {type:"integer"}
#@markdown **每批训练样本数 · BATCH_SIZE**：0 为自动选择，A100 使用 8，其他 GPU 使用 4；显存不足时可手动调低。
BATCH_SIZE = 0 #@param {type:"integer"}
#@markdown **权重保存间隔 · SAVE_EVERY**：默认每 5 轮；实际取训练轮数与此值的最大公约数，确保最终权重保存。
SAVE_EVERY = 5 #@param {type:"integer"}
#@markdown **完成后自动断开并删除运行时 · AUTO_DELETE_RUNTIME**：默认不勾选。开启后，成功导出的完整 ZIP 会保存到 Drive 并校验，等待 Drive 上传完成后再释放运行时；错误或临时存储情况下保留运行时。
AUTO_DELETE_RUNTIME = False #@param {type:"boolean"}

from pathlib import Path
from google.colab import drive
import os, re, json, math, hashlib, shutil, subprocess, zipfile

def mydrive_relative(value):
    value = value.strip().replace('\\', '/')
    # 兼容复制的旧完整路径、MyDrive 前缀及开头的斜杠。
    if value.startswith('/content/drive/MyDrive/'):
        value = value[len('/content/drive/MyDrive/'):]
    value = value.lstrip('/')
    if value == 'MyDrive':
        value = ''
    elif value.startswith('MyDrive/'):
        value = value[len('MyDrive/'):]
    assert value and ':' not in value and '..' not in Path(value).parts, '请填写 MyDrive 下的相对路径，不要填写分享链接或电脑磁盘路径'
    return Path(value).as_posix()
assert DATASET_ZIP.strip(), '请填写 ZIP 的 MyDrive 相对路径，例如 GPT_SoVITS/voice_dataset.zip'
DATASET_ZIP = mydrive_relative(DATASET_ZIP)
assert Path(DATASET_ZIP).suffix.lower() == '.zip', '素材包必须是 ZIP'
auto_name = re.sub(r'[^A-Za-z0-9_-]+', '_', Path(DATASET_ZIP).stem).strip('_-')[:64]
auto_name = auto_name or ('voice_' + hashlib.sha256(Path(DATASET_ZIP).stem.encode('utf-8')).hexdigest()[:8])
EXP_NAME = EXPERIMENT_NAME.strip() or auto_name
assert re.fullmatch(r'[A-Za-z0-9][A-Za-z0-9_-]{0,63}', EXP_NAME), '自定义实验名请使用 1–64 位英文字母、数字、下划线或短横线'
assert EPOCHS_S2 > 0 and EPOCHS_S1 > 0 and BATCH_SIZE >= 0 and SAVE_EVERY > 0, '训练参数必须为有效正数，batch 0 表示自动'
DRIVE_FOLDER = mydrive_relative(DRIVE_FOLDER)
gpu_name = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True).strip()
assert gpu_name, '请启用 Colab GPU'
BATCH = BATCH_SIZE or (8 if 'A100' in gpu_name else 4)
print('实验:', EXP_NAME, 'GPU:', gpu_name, 'batch:', BATCH)
DRIVE_READY = False
if USE_DRIVE:
    try:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=180000)
        DRIVE_READY = Path('/content/drive/MyDrive').is_dir()
    except Exception as exc:
        print('Drive 挂载失败:', exc)
MY_DRIVE = Path('/content/drive/MyDrive')
DRIVE_BASE = MY_DRIVE / DRIVE_FOLDER
STORE = (DRIVE_BASE if DRIVE_READY else Path('/content/GPT_SoVITS_exports')) / EXP_NAME
STORE.mkdir(parents=True, exist_ok=True)
print('导出与备份目录:', STORE)
if not DRIVE_READY:
    print('使用临时存储，训练完成后请下载 ZIP。')

REVISION = '1b47ca07ef43a284e0b8bd7f29d3e1e3ff76082a'
ROOT = Path('/content/GPT-SoVITS')
WORK = ROOT / 'output' / EXP_NAME
RAW = WORK / 'raw'
VOCALS = WORK / 'vocals'
ASR_OUT = WORK / 'asr'
LIST = ASR_OUT / 'vocals.list'
EXP = ROOT / 'logs' / EXP_NAME
LOG_DIR = Path('/content/GPT_SoVITS_runs') / EXP_NAME / 'logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)
CONDA_ROOT = Path('/content/miniforge3')
CONDA_BIN = CONDA_ROOT / 'bin/conda'
PY = CONDA_ROOT / 'envs/GPTSoVITS/bin/python'
EXPORT = STORE / 'weights'
ARCHIVE = Path('/content') / f'{EXP_NAME}_weights.zip'
S2_CONFIG = EXP / 'train_s2.json'
S1_CONFIG = EXP / 'train_s1.yaml'
PRE = ROOT / 'GPT_SoVITS/pretrained_models'
BERT = PRE / 'chinese-roberta-wwm-ext-large'
HUBERT = PRE / 'chinese-hubert-base'
G = PRE / 'gsv-v2final-pretrained/s2G2333k.pth'
D = PRE / 'gsv-v2final-pretrained/s2D2333k.pth'
S1 = PRE / 'gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt'
RUN_CONFIG = dict(exp_name=EXP_NAME, raw=str(RAW), vocals=str(VOCALS), label=str(LIST), exp=str(EXP), store=str(STORE), root=str(ROOT), separate_bgm=SEPARATE_BGM)

def child_environment(extra=None):
    result = os.environ.copy()
    result.update({'PYTHONUNBUFFERED': '1', 'MPLBACKEND': 'agg', 'RUN_CONFIG': json.dumps(RUN_CONFIG), 'version': 'v2', '_CUDA_VISIBLE_DEVICES': '0'})
    result['PATH'] = str(CONDA_ROOT / 'bin') + os.pathsep + result.get('PATH', '')
    paths = [ROOT, ROOT/'GPT_SoVITS', ROOT/'GPT_SoVITS/BigVGAN', ROOT/'tools']
    result['PYTHONPATH'] = os.pathsep.join([str(p) for p in paths] + [result.get('PYTHONPATH', '')])
    if extra:
        result.update({key: str(value) for key, value in extra.items()})
    return result

def run_command(command, name, cwd=None, extra=None):
    # 追加日志，保留前次失败信息。
    log_path = LOG_DIR / (name + '.log')
    print('执行:', name, '日志:', log_path, flush=True)
    with log_path.open('a', encoding='utf-8') as log:
        log.write('\n--- 本次执行 ---\n')
        with subprocess.Popen([str(x) for x in command], cwd=cwd, env=child_environment(extra), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace', bufsize=1) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                log.write(line)
            status = proc.wait()
    if status:
        raise RuntimeError(f'{name} 退出码 {status}，具体错误见上方和 {log_path}')

def run_python(source, name, extra=None):
    assert PY.is_file(), '请先完成环境安装'
    run_command([PY, '-u', '-c', source], name, cwd=ROOT, extra=extra)

def run_gsv(script, args=(), extra=None):
    (ROOT / 'TEMP').mkdir(exist_ok=True)
    additions = {'TEMP': ROOT/'TEMP'}
    if extra:
        additions.update(extra)
    run_command([PY, '-u', '-s', ROOT/script, *args], Path(script).stem, cwd=ROOT, extra=additions)

def finish_runtime_after_export(export_complete):
    if not AUTO_DELETE_RUNTIME:
        print('自动结束运行时未开启，连接保持。')
        return False
    if not export_complete:
        print('导出尚未成功完成，保留运行时。')
        return False
    if not DRIVE_READY:
        print('结果仅在临时目录中，自动结束已取消。请下载 ZIP，运行时保留。')
        return False
    def digest(path):
        hasher = hashlib.sha256()
        with path.open('rb') as stream:
            for chunk in iter(lambda: stream.read(8*1024*1024), b''):
                hasher.update(chunk)
        return hasher.hexdigest()
    pending = STORE/(ARCHIVE.name+'.tmp')
    try:
        from google.colab import runtime
        assert STORE.resolve().is_relative_to(MY_DRIVE.resolve()), '备份目录不在 MyDrive 内'
        assert ARCHIVE.is_file(), '没有找到完整导出 ZIP'
        with zipfile.ZipFile(ARCHIVE) as bundle:
            assert bundle.testzip() is None, '导出 ZIP 完整性检查失败'
        expected_digest = digest(ARCHIVE)
        saved_zip = STORE/ARCHIVE.name
        print('将完整导出 ZIP 保存到 Drive:', saved_zip, flush=True)
        shutil.copy2(ARCHIVE, pending)
        assert pending.stat().st_size == ARCHIVE.stat().st_size and digest(pending) == expected_digest, 'Drive ZIP 保存校验失败'
        pending.replace(saved_zip)
        receipt = {'experiment': EXP_NAME, 'archive': saved_zip.name, 'sha256': expected_digest, 'bytes': saved_zip.stat().st_size, 'status': 'ready_to_release'}
        (STORE/'runtime_completion.json').write_text(json.dumps(receipt, ensure_ascii=False, indent=2), encoding='utf-8')
        print('ZIP 校验通过，等待 Drive 写入云端完成；失败时保留运行时。', flush=True)
        drive.flush_and_unmount(timeout_ms=600000)
        print('Drive 写入完成，正在自动断开并释放运行时。', flush=True)
        runtime.unassign()
        return True
    except Exception as exc:
        print('自动结束未执行或未完成，运行时不会被强制关闭:', str(exc), flush=True)
        return False
    finally:
        # Drive 可能已经卸载，此后不再尝试写入 Drive。
        try:
            if Path('/content/drive/MyDrive').is_dir():
                pending.unlink(missing_ok=True)
        except OSError:
            pass


In [ ]:
#@title 1.1 Drive 缓存工具：第一步后运行，环境和模型跨素材实验复用
import platform, tarfile, uuid
from pathlib import PurePosixPath

CACHE_ID = {'schema': 1, 'revision': REVISION, 'torch': '2.8.0+cu126', 'torchcodec': '0.7.0', 'python': '3.10', 'platform': platform.system(), 'arch': platform.machine(), 'prefix': str(CONDA_ROOT)}
CACHE_KEY = 'v2-cu126-py310-' + hashlib.sha256(json.dumps(CACHE_ID, sort_keys=True).encode()).hexdigest()[:12]
CACHE_DIR = DRIVE_BASE / 'cache' / CACHE_KEY if DRIVE_READY and USE_DRIVE_CACHE else None
CACHE_WORK = Path('/content/.gpt_sovits_cache_work')
CACHE_WORK.mkdir(parents=True, exist_ok=True)
BAD_CACHES = set()
ASR_MODELS = ROOT / 'tools/asr/models'
UVR_MODELS = ROOT / 'tools/uvr5/model_cache'
G2PW_MODELS = ROOT / 'GPT_SoVITS/text/G2PWModel'
RUN_CONFIG['uvr_models'] = str(UVR_MODELS)

def cache_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8*1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def cache_paths(name):
    return CACHE_DIR/(name+'.tar.gz'), CACHE_DIR/(name+'.json')

def restore_cache(name, target):
    if CACHE_DIR is None:
        return False
    archive, metadata = cache_paths(name)
    if not archive.is_file() or not metadata.is_file():
        return False
    local = CACHE_WORK/(name+'.restore.tar.gz')
    previous = None
    extraction_started = False
    try:
        info = json.loads(metadata.read_text(encoding='utf-8'))
        if info.get('cache_id') != CACHE_ID or info.get('root_name') != target.name:
            return False
        print('从 Drive 恢复缓存:', name, flush=True)
        shutil.copy2(archive, local)
        assert cache_sha256(local) == info['sha256'], '缓存完整性校验失败'
        assert target.resolve().is_relative_to(Path('/content').resolve()), '恢复目录必须位于 /content 内'
        with tarfile.open(local, 'r:gz') as bundle:
            for member in bundle.getmembers():
                entry = PurePosixPath(member.name)
                assert not entry.is_absolute() and '..' not in entry.parts and entry.parts[0] == target.name, '缓存包含非法路径'
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                # 保留旧目录，避免把不同依赖混进同一个环境。
                previous = target.with_name(target.name+'.before_cache_'+uuid.uuid4().hex[:8])
                target.rename(previous)
            extraction_started = True
            bundle.extractall(target.parent, filter='data')
        print('缓存恢复完成:', target, flush=True)
        return True
    except Exception as exc:
        if extraction_started:
            if target.exists():
                target.rename(target.with_name(target.name+'.failed_cache_'+uuid.uuid4().hex[:8]))
            if previous is not None and previous.exists():
                previous.rename(target)
        BAD_CACHES.add(name)
        print('缓存未能恢复，将使用正常安装或下载流程:', name, str(exc), flush=True)
        return False
    finally:
        local.unlink(missing_ok=True)

def save_cache(name, source):
    if CACHE_DIR is None or not source.is_dir():
        return
    assert source.resolve().is_relative_to(Path('/content').resolve()), '缓存来源必须位于 /content 内'
    archive, metadata = cache_paths(name)
    if name not in BAD_CACHES and archive.is_file() and metadata.is_file():
        try:
            info = json.loads(metadata.read_text(encoding='utf-8'))
            if info.get('cache_id') == CACHE_ID and info.get('root_name') == source.name:
                return
        except (ValueError, OSError):
            pass
    print('首次保存 Drive 缓存:', name, '打包需要一些时间', flush=True)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    local = CACHE_WORK/(name+'.save.tar.gz')
    pending = archive.with_name(archive.name+'.tmp')
    pending_meta = metadata.with_name(metadata.name+'.tmp')
    def archive_filter(member):
        # Conda 下载包缓存可重新生成，不必把它重复打进环境包。
        parts = PurePosixPath(member.name).parts
        if name == 'environment' and len(parts) > 1 and parts[1] == 'pkgs':
            return None
        return member
    try:
        with tarfile.open(local, 'w:gz', compresslevel=1) as bundle:
            bundle.add(source, arcname=source.name, filter=archive_filter)
        info = {'cache_id': CACHE_ID, 'root_name': source.name, 'sha256': cache_sha256(local), 'bytes': local.stat().st_size}
        shutil.copy2(local, pending)
        assert cache_sha256(pending) == info['sha256'], 'Drive 缓存写入校验失败'
        pending_meta.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding='utf-8')
        pending.replace(archive)
        pending_meta.replace(metadata)
        BAD_CACHES.discard(name)
        print('缓存已保存:', archive, flush=True)
    finally:
        local.unlink(missing_ok=True)
        pending.unlink(missing_ok=True)
        pending_meta.unlink(missing_ok=True)

if CACHE_DIR is not None:
    print('跨实验共用的 Drive 缓存目录:', CACHE_DIR)
else:
    print('Drive 缓存未启用；仅复用当前会话的本地环境。')


## 2. 安装固定版本环境

首次安装独立 Python 3.10 环境，固定 torch/torchaudio/torchvision 的 CUDA 构建，并使用 `agg` 绘图后端。成功后将环境及预训练模型压缩保存到 `MyDrive/<DRIVE_FOLDER>/cache/`。后续会话优先恢复到 `/content`，再验证 ASR、训练及音频分离的关键导入；缓存失效或版本变化时重新安装。

首次缓存会占用额外 Drive 空间并花时间打包，之后仍需解压和校验。运行时、CUDA 驱动和当前 GPU 由 Colab 提供，环境缓存不能替代它们。

In [ ]:
if not ROOT.exists():
    run_command(['git', 'clone', 'https://github.com/RVC-Boss/GPT-SoVITS.git', ROOT], 'git_clone')
if subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip() != REVISION:
    run_command(['git', '-C', ROOT, 'checkout', REVISION], 'git_revision')
setup_marker = CONDA_ROOT / 'gpt_sovits_setup.json'
setup_id = {'revision': REVISION, 'torch': '2.8.0+cu126', 'torchcodec': '0.7.0'}
def local_setup_ready():
    try:
        return PY.is_file() and setup_marker.is_file() and json.loads(setup_marker.read_text()) == setup_id
    except (ValueError, OSError):
        return False
if not local_setup_ready():
    restore_cache('environment', CONDA_ROOT)
if not all(path.exists() for path in [BERT, HUBERT, G, D, S1]):
    restore_cache('pretrained_models', PRE)
if not CONDA_BIN.is_file():
    installer = Path('/content/Miniforge3-Linux-x86_64.sh')
    run_command(['curl', '-fL', '--retry', '3', '-o', installer, 'https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh'], 'miniforge_download')
    run_command(['bash', installer, '-b', '-p', CONDA_ROOT], 'miniforge_install')
if not PY.is_file():
    run_command([CONDA_BIN, 'create', '-n', 'GPTSoVITS', 'python=3.10', '-y'], 'conda_create')
constraints = Path('/content/gpt_sovits_torch_constraints.txt')
constraints.write_text('torch==2.8.0+cu126\ntorchaudio==2.8.0+cu126\ntorchvision==0.23.0+cu126\ntorchcodec==0.7.0\n', encoding='utf-8')
if not local_setup_ready() or not all(path.exists() for path in [BERT, HUBERT, G, D, S1]):
    run_command([CONDA_BIN, 'run', '--no-capture-output', '-n', 'GPTSoVITS', 'bash', 'install.sh', '--device', 'CU126', '--source', 'HF'], 'official_install', cwd=ROOT)
    index = 'https://' + 'download.pytorch.org/whl/cu126'
    run_command([PY, '-m', 'pip', 'install', '--upgrade', 'torch==2.8.0+cu126', 'torchaudio==2.8.0+cu126', 'torchvision==0.23.0+cu126', '--index-url', index], 'pytorch_install')
    run_command([PY, '-m', 'pip', 'install', '--upgrade', '--no-deps', 'torchcodec==0.7.0'], 'torchcodec_install')
    run_command([PY, '-m', 'pip', 'install', 'audio-separator[gpu]', '-c', constraints, '--extra-index-url', index], 'separator_install')
verify = '''
import torch, torchaudio, torchvision, pytorch_lightning
from funasr import AutoModel
from audio_separator.separator import Separator
assert torch.__version__ == torchaudio.__version__ == '2.8.0+cu126'
assert torchvision.__version__ == '0.23.0+cu126'
assert torch.cuda.is_available(), '训练环境无法使用 GPU'
print('环境检查通过:', torch.__version__, torchaudio.__version__, torch.cuda.get_device_name(0))
'''
run_python(verify, 'environment_check')
for path in [BERT, HUBERT, G, D, S1]:
    assert path.exists(), f'缺少 v2 预训练模型: {path}'
setup_marker.write_text(json.dumps(setup_id))
save_cache('environment', CONDA_ROOT)
save_cache('pretrained_models', PRE)
for name, target in [('asr_models', ASR_MODELS), ('uvr_models', UVR_MODELS), ('g2pw_models', G2PW_MODELS)]:
    # 分阶段首次成功后保存，后续新会话提前恢复，避免重复下载。
    has_weights = target.is_dir() and any(p.is_file() for pattern in ['*.pt', '*.pth', '*.bin', '*.ckpt', '*.onnx', '*.safetensors'] for p in target.rglob(pattern))
    if not has_weights:
        restore_cache(name, target)
(STORE / 'gpt_sovits_revision.txt').write_text(REVISION)


## 3. 导入素材包

Drive 已连接时，直接读取 `MyDrive/<DATASET_ZIP>`。找不到则显示完整查找路径，修改第一格并重跑即可。Drive 未连接时，优先查找 `/content/<ZIP文件名>`；不存在才弹出上传窗口。支持 ZIP 内有子目录，同名 WAV 会提前报错。

In [ ]:
requested = Path(DATASET_ZIP)
if DRIVE_READY:
    ZIP = MY_DRIVE / requested
    assert ZIP.is_file(), f'素材文件不存在: {ZIP}。请修改第一格的 DATASET_ZIP，它相对于 MyDrive；不要在路径中重复备份目录。'
else:
    candidates = [Path('/content')/requested.name, STORE/requested.name]
    ZIP = next((p for p in candidates if p.is_file()), None)
if ZIP is None:
    from google.colab import files
    print('请上传:', requested.name)
    uploaded = files.upload()
    assert requested.name in uploaded, f'上传文件名应为 {requested.name}'
    ZIP = Path('/content') / requested.name
    ZIP.write_bytes(uploaded[requested.name])
def sha256_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024*1024), b''):
            h.update(chunk)
    return h.hexdigest()
manifest = {'dataset_zip': ZIP.name, 'dataset_sha256': sha256_file(ZIP), 'separate_bgm': SEPARATE_BGM, 'revision': REVISION}
WORK.mkdir(parents=True, exist_ok=True)
manifest_path = WORK / 'dataset_manifest.json'
saved_manifest = STORE / 'dataset_manifest.json'
if saved_manifest.exists():
    assert json.loads(saved_manifest.read_text(encoding='utf-8')) == manifest, '备份目录已有其他素材或分离配置；请在开头使用新的实验名称'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, '此实验已有不同素材或分离配置；请在开头使用新的实验名称后重跑'
elif RAW.exists() and any(RAW.iterdir()):
    raise RuntimeError('实验目录已有来源未知的音频；请更换实验名称')
RAW.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP) as z:
    items = [x for x in z.infolist() if not x.is_dir() and x.filename.lower().endswith('.wav') and not x.filename.startswith('__MACOSX/')]
    assert items, 'ZIP 中没有 WAV；请先将录播切成短句 WAV'
    def wav_name(item):
        return Path(item.filename.replace('\\', '/')).stem + '.wav'
    names = [wav_name(x) for x in items]
    assert len(names) == len({name.casefold() for name in names}), 'ZIP 中有重名 WAV，请先重命名'
    # 解压前记录来源，解压中断后可安全续跑同一个包。
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
    for item in items:
        target = RAW / wav_name(item)
        if not target.is_file() or target.stat().st_size != item.file_size:
            with z.open(item) as src, target.open('wb') as dst:
                shutil.copyfileobj(src, dst)
(STORE / 'dataset_manifest.json').write_text(manifest_path.read_text(encoding='utf-8'), encoding='utf-8')
print('素材包:', ZIP, 'WAV 数量:', len(items))


In [ ]:
#@title 4. 分离 BGM；已清洗素材可在开头关闭 SEPARATE_BGM
separate_script = '''
from pathlib import Path
import os, json, shutil
import soundfile as sf
cfg = json.loads(os.environ['RUN_CONFIG'])
raw, out = Path(cfg['raw']), Path(cfg['vocals'])
tmp = out.parent / 'separation'
out.mkdir(parents=True, exist_ok=True)
tmp.mkdir(parents=True, exist_ok=True)
files = sorted(raw.glob('*.wav'))
def valid(path):
    try:
        info = sf.info(str(path))
        return info.frames > 0 and info.samplerate > 0
    except Exception:
        return False
pending = [src for src in files if not valid(out / src.name)]
sep = None
if pending and cfg['separate_bgm']:
    from audio_separator.separator import Separator
    sep = Separator(output_dir=str(tmp), output_format='WAV', sample_rate=44100, model_file_dir=cfg['uvr_models'])
    sep.load_model(model_filename='model_bs_roformer_ep_317_sdr_12.9755.ckpt')
for i, src in enumerate(pending, 1):
    if sep is None:
        shutil.copy2(src, out/src.name)
    else:
        outputs = sep.separate(str(src))
        vocals = [tmp/Path(x).name for x in outputs if '(vocals)' in Path(x).stem.lower()]
        assert len(vocals) == 1 and valid(vocals[0]), f'人声输出无效: {src.name}: {outputs}'
        shutil.copy2(vocals[0], out/src.name)
    assert valid(out/src.name), f'音频无效: {src.name}'
    print(f'{i}/{len(pending)} {src.name}', flush=True)
assert {x.name for x in out.glob('*.wav')} == {x.name for x in files}, '分离数量不一致'
print('清洗音频:', out, '数量:', len(files))
'''
run_python(separate_script, 'separation')
if SEPARATE_BGM:
    save_cache('uvr_models', UVR_MODELS)
shutil.copytree(VOCALS, STORE/'dataset_backup/cleaned_wavs', dirs_exist_ok=True)
print('清洗 WAV 已备份:', STORE/'dataset_backup/cleaned_wavs')


## 5. 中文 ASR、自动重试与标注核对

默认使用中文 Paraformer。已有标注会保留；空文本或漏识别会只针对相应文件关闭 VAD 重试一次，打印重试文本供核对。仍为空则停止并列出文件名。无需逐条播放，但请抽查识别文字，尤其是重试结果。

In [ ]:
asr_script = r'''
from pathlib import Path
import os, json, shutil
import torch
from funasr import AutoModel
from tools.asr.funasr_asr import execute_asr
cfg = json.loads(os.environ['RUN_CONFIG'])
root, vocals, label = Path(cfg['root']), Path(cfg['vocals']), Path(cfg['label'])
label.parent.mkdir(parents=True, exist_ok=True)
if not label.exists():
    execute_asr(str(vocals), str(label.parent), 'large', 'zh', backend='classic')
rows = [line.split('|', 3) for line in label.read_text(encoding='utf-8').splitlines() if line.strip()]
assert all(len(row) == 4 for row in rows), '标注格式错误，请检查 vocals.list'
files = sorted(vocals.glob('*.wav'))
by_name = {Path(row[0]).name: row for row in rows}
assert len(by_name) == len(rows), '标注含重复音频'
assert set(by_name).issubset({f.name for f in files}), '标注包含未知音频'
for audio in files:
    row = by_name.setdefault(audio.name, [str(audio), cfg['exp_name'], 'ZH', ''])
    row[0] = str(audio)
pending = [audio for audio in files if not by_name[audio.name][3].strip()]
report = []
if pending:
    print('关闭 VAD 重试:', [p.name for p in pending], flush=True)
    model_dir = root/'tools/asr/models/speech_paraformer-large_asr_nat-zh-cn-16k-common-vocab8404-pytorch'
    model = AutoModel(model=str(model_dir), device='cuda' if torch.cuda.is_available() else 'cpu', disable_update=True)
    for audio in pending:
        result = model.generate(input=str(audio))
        text = str(result[0].get('text', '')).strip() if result else ''
        text = ' '.join(text.split()).replace('|', '｜')
        by_name[audio.name][3] = text
        report.append({'file': audio.name, 'text_without_vad': text})
        print(audio.name, repr(text), flush=True)
    (label.parent/'asr_retry_without_vad.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
shutil.copy2(label, label.with_suffix('.list.bak'))
rows = [by_name[audio.name] for audio in files]
for row in rows:
    row[3] = ' '.join(row[3].split()).replace('|', '｜')
temp = label.with_suffix('.list.tmp')
temp.write_text('\n'.join('|'.join(row) for row in rows) + '\n', encoding='utf-8')
temp.replace(label)
empty = [Path(row[0]).name for row in rows if not row[3].strip()]
assert not empty, f'重试后仍为空: {empty}。只需检查这些片段；已有标注已保留。'
print('非空标注检查通过:', len(rows), flush=True)
for row in rows[:10]:
    print(Path(row[0]).name, row[3], flush=True)
'''
run_python(asr_script, 'asr')
save_cache('asr_models', ASR_MODELS)
print('标注文件:', LIST)
print('如需改字，可在 Colab 文件栏编辑此文件；下方继续运行会使用修订后的标注。')


In [ ]:
#@title 6. 核对后备份标注
shutil.copy2(LIST, STORE/'dataset_backup/vocals.list')
print('标注已备份:', STORE/'dataset_backup/vocals.list')


## 7. 特征提取

每一步都检查全部音频的输出。部分输出不完整时可重跑此格，会保留已生成的有效特征。已经提取特征后再修改标注时，请使用新的实验名重新训练，避免文本与缓存错配。

打印“特征检查通过”表示提取与文件检查已完成。末尾保存 G2PW 模型到 Drive 属于可选缓存；缓存工具未初始化或写入失败时会提示原因，可继续第八步。需要启用缓存时，请先运行第一步后面的“1.1 Drive 缓存工具”单元格。

In [ ]:
EXP.mkdir(parents=True, exist_ok=True)
labels = [line.split('|') for line in LIST.read_text(encoding='utf-8').splitlines() if line.strip()]
assert labels and all(len(row) == 4 and row[3].strip() for row in labels), '标注含空文本或格式错误'
expected = {Path(row[0]).name for row in labels}
assert len(expected) == len(labels), '标注存在重名音频'
feature_id = {'manifest': manifest, 'labels_sha256': sha256_file(LIST)}
feature_marker = EXP/'feature_inputs.json'
if feature_marker.exists():
    assert json.loads(feature_marker.read_text()) == feature_id, '特征缓存对应不同标注；请使用新的实验名重新运行'
feature_marker.write_text(json.dumps(feature_id, ensure_ascii=False, indent=2), encoding='utf-8')
prepare_env = {'inp_text': LIST, 'inp_wav_dir': VOCALS, 'exp_name': EXP_NAME, 'opt_dir': EXP, 'i_part': 0, 'all_parts': 1, 'is_half': 'True', 'bert_pretrained_dir': BERT, 'cnhubert_base_dir': HUBERT, 'pretrained_s2G': G, 's2config_path': ROOT/'GPT_SoVITS/configs/s2.json'}
def check_table(path):
    lines = [line for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    names = [line.split('\t', 1)[0] for line in lines]
    assert set(names) == expected and len(names) == len(expected), f'{path.name} 不完整，缺少: {sorted(expected-set(names))}，请检查对应日志'
    return '\n'.join(lines) + '\n'
for name in ['2-name2text-0.txt', '6-name2semantic-0.tsv']:
    part = EXP/name
    if part.exists():
        try:
            check_table(part)
        except AssertionError:
            shutil.copy2(part, part.with_suffix(part.suffix+'.incomplete.bak'))
            part.unlink()
run_gsv('GPT_SoVITS/prepare_datasets/1-get-text.py', extra=prepare_env)
(EXP/'2-name2text.txt').write_text(check_table(EXP/'2-name2text-0.txt'), encoding='utf-8')
missing = [Path(row[0]).name for row in labels if row[2].lower() == 'zh' and not (EXP/'3-bert'/(Path(row[0]).name+'.pt')).is_file()]
assert not missing, f'BERT 特征缺少: {missing}'
run_gsv('GPT_SoVITS/prepare_datasets/2-get-hubert-wav32k.py', extra=prepare_env)
for folder, suffix in [('4-cnhubert', '.pt'), ('5-wav32k', '')]:
    missing = [name for name in sorted(expected) if not (EXP/folder/(name+suffix)).is_file()]
    assert not missing, f'{folder} 缺少: {missing}，请检查日志'
run_gsv('GPT_SoVITS/prepare_datasets/3-get-semantic.py', extra=prepare_env)
(EXP/'6-name2semantic.tsv').write_text('item_name\tsemantic_audio\n'+check_table(EXP/'6-name2semantic-0.tsv'), encoding='utf-8')
print('特征检查通过:', len(expected))
# 可选模型缓存不应阻断已完成的特征提取；兼容更新 Notebook 后的旧会话。
cache_writer = globals().get('save_cache')
if callable(cache_writer):
    try:
        cache_writer('g2pw_models', ROOT/'GPT_SoVITS/text/G2PWModel')
    except Exception as exc:
        print('G2PW 模型缓存保存失败:', str(exc), '。特征已完成，可继续第八步。')
else:
    print('跳过 G2PW 模型缓存：请先运行“1.1 Drive 缓存工具”单元格以启用缓存。特征已完成，可继续第八步。')


## 8. 生成训练配置

轮数、batch 和保存间隔使用开头参数。实际保存间隔取轮数与指定间隔的最大公约数，保证最后一轮一定导出模型。训练前创建检查点与权重目录。

In [ ]:
train_options = {'batch': BATCH, 'epochs_s2': EPOCHS_S2, 'epochs_s1': EPOCHS_S1, 'save_s2': math.gcd(EPOCHS_S2, SAVE_EVERY), 'save_s1': math.gcd(EPOCHS_S1, SAVE_EVERY), 'g': str(G), 'd': str(D), 's1': str(S1)}
config_script = '''
from pathlib import Path
import os, json, yaml
cfg = json.loads(os.environ['RUN_CONFIG'])
opt = json.loads(os.environ['TRAIN_OPTIONS'])
root, exp = Path(cfg['root']), Path(cfg['exp'])
name = cfg['exp_name']
for folder in [exp/'logs_s2_v2', exp/'logs_s1_v2/ckpt', root/'SoVITS_weights_v2', root/'GPT_weights_v2']:
    folder.mkdir(parents=True, exist_ok=True)
s2 = json.loads((root/'GPT_SoVITS/configs/s2.json').read_text())
s2['train'].update(batch_size=opt['batch'], epochs=opt['epochs_s2'], pretrained_s2G=opt['g'], pretrained_s2D=opt['d'], if_save_latest=True, if_save_every_weights=True, save_every_epoch=opt['save_s2'], gpu_numbers='0')
s2['model']['version'] = 'v2'
s2['data']['exp_dir'] = s2['s2_ckpt_dir'] = str(exp)
s2.update(save_weight_dir=str(root/'SoVITS_weights_v2'), name=name, version='v2')
(exp/'train_s2.json').write_text(json.dumps(s2, ensure_ascii=False, indent=2), encoding='utf-8')
s1 = yaml.safe_load((root/'GPT_SoVITS/configs/s1longer-v2.yaml').read_text())
s1['train'].update(batch_size=opt['batch'], epochs=opt['epochs_s1'], save_every_n_epoch=opt['save_s1'], if_save_every_weights=True, if_save_latest=True, if_dpo=False, half_weights_save_dir=str(root/'GPT_weights_v2'), exp_name=name)
s1.update(pretrained_s1=opt['s1'], train_semantic_path=str(exp/'6-name2semantic.tsv'), train_phoneme_path=str(exp/'2-name2text.txt'), output_dir=str(exp/'logs_s1_v2'))
(exp/'train_s1.yaml').write_text(yaml.safe_dump(s1, allow_unicode=True), encoding='utf-8')
print('配置生成完成:', exp, '保存间隔:', opt['save_s2'], opt['save_s1'])
'''
run_python(config_script, 'training_config', extra={'TRAIN_OPTIONS': json.dumps(train_options)})
def backup_weights(folder, pattern, stage):
    target = EXPORT/folder
    target.mkdir(parents=True, exist_ok=True)
    exact = rf'{re.escape(EXP_NAME)}_e[0-9]+_s[0-9]+\.pth' if folder == 'SoVITS_weights_v2' else rf'{re.escape(EXP_NAME)}-e[0-9]+\.ckpt'
    found = sorted(p for p in (ROOT/folder).glob(pattern) if re.fullmatch(exact, p.name))
    assert found, f'{stage} 未找到推理权重: {pattern}'
    for p in found:
        shutil.copy2(p, target/p.name)
    print(stage, '权重已备份:', target)


In [ ]:
#@title 9. SoVITS 训练，完成后立即备份声学权重
(EXP/'logs_s2_v2').mkdir(parents=True, exist_ok=True)
(ROOT/'SoVITS_weights_v2').mkdir(exist_ok=True)
run_gsv('GPT_SoVITS/s2_train.py', args=('--config', str(S2_CONFIG)))
backup_weights('SoVITS_weights_v2', f'{EXP_NAME}_*.pth', 'SoVITS')


In [ ]:
#@title 10. GPT 训练，完成后立即备份韵律权重
(EXP/'logs_s1_v2/ckpt').mkdir(parents=True, exist_ok=True)
(ROOT/'GPT_weights_v2').mkdir(exist_ok=True)
# Lightning 的训练检查点包含 PosixPath；PyTorch 2.6+ 的安全加载需要显式允许。
# 必须在独立训练进程中注册，在 Colab 主内核中注册不会传入子进程。
gpt_train_script = '''
import runpy, sys
from pathlib import PosixPath
import torch
torch.serialization.add_safe_globals([PosixPath])
script = sys.argv[1]
sys.argv = sys.argv[1:]
runpy.run_path(script, run_name='__main__')
'''
gpt_runner = LOG_DIR/'s1_train_resume.py'
gpt_runner.write_text(gpt_train_script, encoding='utf-8')
(ROOT/'TEMP').mkdir(exist_ok=True)
run_command([PY, '-u', '-s', gpt_runner, ROOT/'GPT_SoVITS/s1_train.py', '--config_file', str(S1_CONFIG)], 's1_train', cwd=ROOT, extra={'hz': '25hz', 'MPLBACKEND': 'agg', 'TEMP': ROOT/'TEMP'})
backup_weights('GPT_weights_v2', f'{EXP_NAME}-*.ckpt', 'GPT')


## 11. 完整导出并给出下载位置

导出包包含两个权重目录、配置、清洗后的全部 WAV、每个 WAV 的准确标注映射、数据来源记录和日志。参考音频候选按 3–10 秒筛选；请试听确认候选句子是否适合日常朗读。无论 Drive 是否连接，都会生成可下载的 ZIP。

开启 `AUTO_DELETE_RUNTIME` 时，本格最后会额外把完整 ZIP 保存到 Drive，并在完整性校验及 Drive 同步完成后调用 Colab 的运行时释放接口。默认关闭；前面的训练、导出或同步失败时不释放。用于无人值守时，请先完成 Drive 授权并从头按顺序运行到本格。

In [ ]:
EXPORT_COMPLETE = False
backup_weights('SoVITS_weights_v2', f'{EXP_NAME}_*.pth', 'SoVITS')
backup_weights('GPT_weights_v2', f'{EXP_NAME}-*.ckpt', 'GPT')
# 重跑配置可能改变 STORE；导出时根据当前源码重建版本记录。
export_revision = subprocess.check_output(['git', '-C', str(ROOT), 'rev-parse', 'HEAD'], text=True).strip()
assert export_revision == REVISION, f'源码提交与固定版本不一致: {export_revision}，请核对环境后再导出'
revision_path = STORE/'gpt_sovits_revision.txt'
revision_path.write_text(export_revision, encoding='utf-8')
for source in [S1_CONFIG, S2_CONFIG, LIST, manifest_path, feature_marker, revision_path]:
    shutil.copy2(source, EXPORT/source.name)
shutil.copytree(VOCALS, EXPORT/'reference_audio', dirs_exist_ok=True)
shutil.copytree(LOG_DIR, EXPORT/'logs', dirs_exist_ok=True)
rows = [line.split('|', 3) for line in LIST.read_text(encoding='utf-8').splitlines() if line.strip()]
references = {Path(row[0]).name: row[3] for row in rows}
(EXPORT/'reference_texts.json').write_text(json.dumps(references, ensure_ascii=False, indent=2), encoding='utf-8')
reference_script = '''
from pathlib import Path
import os, json, soundfile as sf
cfg = json.loads(os.environ['RUN_CONFIG'])
out = Path(cfg['store'])/'weights'
texts = json.loads((out/'reference_texts.json').read_text(encoding='utf-8'))
candidates = []
for name, text in texts.items():
    info = sf.info(str(out/'reference_audio'/name))
    duration = info.frames/info.samplerate
    if 3 <= duration <= 10:
        candidates.append({'file': 'reference_audio/'+name, 'seconds': round(duration, 2), 'text': text})
(out/'reference_candidates.json').write_text(json.dumps(candidates, ensure_ascii=False, indent=2), encoding='utf-8')
print('参考音频候选示例:', candidates[:3])
'''
run_python(reference_script, 'reference_candidates')
readme = f'''模型版本: GPT-SoVITS v2
实验: {EXP_NAME}
GPT 权重: GPT_weights_v2/{EXP_NAME}-e{EPOCHS_S1}.ckpt
SoVITS 权重: SoVITS_weights_v2/{EXP_NAME}_e{EPOCHS_S2}_s*.pth
参考音频: reference_audio/；参考文字: reference_texts.json
候选列表: reference_candidates.json
本地官方 WebUI 中加载两个权重，上传参考音频并填入其文字；语言选中文。
源码提交: {export_revision}
'''
(EXPORT/'使用说明.txt').write_text(readme, encoding='utf-8')
assert list((EXPORT/'GPT_weights_v2').glob(f'{EXP_NAME}-e{EPOCHS_S1}.ckpt')), '缺少最终 GPT 权重'
assert list((EXPORT/'SoVITS_weights_v2').glob(f'{EXP_NAME}_e{EPOCHS_S2}_s*.pth')), '缺少最终 SoVITS 权重'
shutil.make_archive(str(ARCHIVE.with_suffix('')), 'zip', root_dir=EXPORT)
print('备份目录:', EXPORT)
print('下载 ZIP:', ARCHIVE, '大小 MB:', round(ARCHIVE.stat().st_size/1e6, 1))
print('在 Colab 左侧文件栏找到上方 ZIP，右键下载。Drive 已连接时备份目录也在 MyDrive 中。')
EXPORT_COMPLETE = True
finish_runtime_after_export(EXPORT_COMPLETE)


## 本地推理

在 Windows 官方整合包中，将导出包内 `GPT_weights_v2/` 和 `SoVITS_weights_v2/` 的文件放进同名目录，再启动官方 WebUI。选择最终权重，并从 `reference_audio/` 中挑一段 3–10 秒的参考 WAV，使用 `reference_texts.json` 中对应的准确文字。AMD CPU 可运行 CPU 推理。

已生成的模型可以反复使用，后续念弹幕由本地 API 自动复用固定的参考音频。训练时的真实耗时和本地合成速度需按所用硬件测量。